# Advanced Reasoning- CoT, Self-Consistency, and Knowing When to Stop

**Module 5 · Lesson 5.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Chain-of-thought - make the thinking visible
- Self-consistency - sample, then vote
- Reasoning models - when the model already thinks
- Tree-of-thought - explore, evaluate, pick
- ReAct and reflexion - reason with tools, and self-correct
- Choose and verify - framework, cost, and chain-of-verification

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The whole game: same model, two answers

> 🧠 **Analogy**
>
> **Picture four problem-solvers.** A **student** who writes out every step of a math problem. A **panel** that solves it several times and takes the majority answer. A **chess player** who reads three moves ahead before committing. A **detective** who thinks, investigates, then thinks again. A **writer** who drafts, critiques, and rewrites.
> Those are the five reasoning patterns in this lesson - chain-of-thought, self-consistency, tree-of-thought, ReAct, and reflexion. Learn each, and you can take the same model from a failing score to a strong one on the same hard problem just by changing how it thinks.
> **Where the analogy breaks down:** unlike people, a 2026 reasoning model has already been trained to think. Narrating the method for it - hand-written "think step by step" - can get in its way. And its visible "thoughts" are not a guaranteed faithful record of what it actually computed.

Every technique here is tested with **real API calls** and paired with a plain-English walkthrough of the code. We use Gemini through the current unified SDK, but every pattern transfers to Claude and GPT.

- **Apply** chain-of-thought and structured CoT (separating reasoning from the answer) to multi-step problems against the live API.

- **Compare** classic-model CoT prompting with native reasoning models, and decide when explicit CoT helps and when it hurts.

- **Construct** the multi-call patterns - self-consistency, tree-of-thought, ReAct, and reflexion - and pick the right one per task.

- **Evaluate** reasoning cost, latency, and reliability, and apply chain-of-verification to cut hallucination.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: one helper we will reuse all lesson

Every example calls this `ask()` helper on the current unified **google-genai** SDK (the older `google.generativeai` package was deprecated in 2025 - [migration guide](https://ai.google.dev/gemini-api/docs/migrate)).

**📝 `setup.py Gemini`** - *API*

In [ ]:
# pip install "google-genai>=1.0.0"
from google import genai
from google.genai import types
import os, re
from collections import Counter

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.3, system: str = None) -> str:
    """One completion. `system` sets a role; `temperature` controls randomness."""
    cfg = types.GenerateContentConfig(temperature=temperature, system_instruction=system)
    try:
        return client.models.generate_content(
            model="gemini-3-flash", contents=prompt, config=cfg).text.strip()
    except Exception as e:
        return f"[API error: {e}]"

print(ask("Reply with one word: ready"))
# Output: Ready

- `genai.Client(...)` - one reusable client (the deprecated package used a global `configure()` plus per-call `GenerativeModel`).

- `system` goes into `GenerateContentConfig(system_instruction=...)` - we use it to set a "careful problem solver" role in the CoT step.

- `temperature` is the lever self-consistency (Step 2) relies on: it must be above 0 to sample different reasoning chains.

- The `try/except` keeps one network blip from crashing a multi-call reasoning loop (some patterns here make 3-9 calls).

---

## 🎯 Concept 1: Chain-of-thought - make the thinking visible

### Chain-of-thought - make the thinking visible

The simplest reasoning upgrade: force the model to show its work.

**School math exams.** "Show your working - you get marks for the steps even if the final number is wrong." Writing each step out makes you catch your own mistakes.

The gain: the same trick works on a model. Forcing intermediate steps turns one risky leap into a chain of small, checkable moves.

**📝 `cot.py Gemini`** - *API*

In [ ]:
problem = "Apples cost 40 each. Weekends: buy 2 get 1 free. Rahul buys 7 on Saturday. Total paid?"

direct = problem + " Give only the final number."
cot    = problem + " Think step by step, then end with 'Answer: <number>'."

print("direct:", ask(direct, temperature=0.0))
print("cot:   ", ask(cot, temperature=0.0))
# Output: direct: 280                          (7 x 40 - ignores the offer)
# Output: cot:    ...every 3 apples cost 2, so 2 are free, pay for 5,
# Output:         5 x 40 = 200. Answer: 200    (correct)

- **Same problem, two prompts.** The only difference is "Give only the final number" versus "Think step by step" - so any change in correctness is caused by the reasoning, not the task.

- **`temperature=0.0`** makes both calls deterministic, so the contrast is repeatable rather than luck.

- **"end with 'Answer: <number>'"** gives you a stable marker to parse the final answer out of the reasoning text.

- **The output comments** show the real behaviour: the direct call leaps to 280; the CoT call reasons through the free apples to 200.

In production you do not want the reasoning mixed into the answer you parse. **Structured CoT** separates them with tags:

**📝 `structured_cot.py`** - *Production*

In [ ]:
prompt = """Compute 18% GST on 3 laptops at 45,000 each (intra-state, split CGST/SGST).
Reason inside <thinking> tags. Put ONLY the final breakdown inside <answer> tags."""

resp = ask(prompt, temperature=0.0)
answer = re.search(r"<answer>(.*?)</answer>", resp, re.DOTALL)
print(answer.group(1).strip() if answer else resp)
# Output: Base 135000 | CGST 9% 12150 | SGST 9% 12150 | Total 159300

- **Reasoning first, answer last.** Putting `<thinking>` before `<answer>` forces the model to work before it commits - the order is load-bearing.

- **The regex pulls only the answer.** Your downstream code parses the clean `<answer>` block and ignores the reasoning - which you can still log for debugging.

- This is the prompt-level version of what native reasoning models (Step 3) and structured output (Lesson 5.5) do for you automatically.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Forcing visible steps turns a risky one-shot leap into a chain of checkable moves - measured gains of roughly 20-40% on multi-step math and logic for non-reasoning models (Kojima 2022, Wei 2022). Structured CoT then separates the reasoning from the answer so production code can parse it. **CoT is the floor of reasoning; the rest of this lesson is when one chain is not enough.**

---

## 🎯 Concept 2: Self-consistency - sample, then vote

### Self-consistency - sample, then vote

One chain can slip. Several chains and a majority vote recover the answer.

**Ask three friends, take the majority.** Any one friend might miscalculate, but it is unlikely all three make the *same* mistake, so the majority answer is more trustworthy than any single one.

The gain: sampling several reasoning chains and voting averages out random slips. It needs temperature > 0 so the chains actually differ.

**📝 `self_consistency.py Gemini`** - *API*

In [ ]:
def self_consistency(problem: str, n: int = 5) -> str:
    """Sample n reasoning chains, return the majority final answer."""
    answers = []
    for _ in range(n):
        chain = ask(problem + " Think step by step, end with 'Answer: <value>'.",
                    temperature=0.7)            # >0 so chains differ
        m = re.search(r"Answer:\s*(.+)", chain)
        if m:
            answers.append(m.group(1).strip().rstrip("."))
    return Counter(answers).most_common(1)[0][0]   # the modal answer

problem = "A train goes 60 km/h for 2.5 h, then 80 km/h for 1.5 h. Total distance?"
print(self_consistency(problem, n=5))
# Output: 270 km     (one or two chains may slip to 300; the majority is 270)

- **The loop samples `n` chains** at `temperature=0.7` - above zero so each chain takes a slightly different path. At temperature 0 you would get the same chain n times and learn nothing.

- **The regex extracts each final answer** from the "Answer:" marker - the same marker trick from Step 1, now used to collect votes.

- **`Counter(...).most_common(1)`** tallies the answers and returns the most frequent one - the majority vote. Production note: unparseable chains are silently dropped (weakening the vote), and an all-unparseable batch makes `most_common(1)[0]` crash on an empty list - real code counts parse failures, requires a minimum quorum, and handles the empty case.

- **The output comment** shows the payoff: individual chains occasionally slip to 300 (a misread of the times), but the correct 270 (60x2.5 + 80x1.5 = 150 + 120) wins the vote.

Self-consistency works best when the answer is *discrete* (a number, a label) so votes can match exactly. For free-form text, there is nothing clean to count.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Self-consistency (Wang et al. 2023) replaces one greedy chain with N sampled chains and a majority vote, trading N-times cost for a large reliability gain on discrete-answer tasks. The catch: it only helps when the errors are *independent* and the right answer is the plurality - on a problem where the model is confidently and consistently wrong, voting just entrenches the wrong answer. **It is the cheapest way to make CoT more trustworthy - when you can afford the samples.**

---

## 🎯 Concept 3: Reasoning models - when the model already thinks

### Reasoning models - when the model already thinks

2026's biggest shift: models that learned to reason, where hand-written CoT can hurt.

**An expert who already shows their work versus a junior you must prompt.** With a junior, "think step by step" earns its keep - it makes them slow down and reason. Say the same to a senior who always works methodically and you just add noise: they were already going to reason, and the instruction can even push them off their own better method.

The gain: a reasoning model is the senior - state the goal and dial the effort, do not narrate the method. Manual CoT is for classic models (Step 1); reasoning models do it internally.

> 📦 **Info**
>
> The 2026 reasoning lineup
> Reasoning is now a model mode, not a prompt trick. **GPT-5.5** exposes a `reasoning_effort` control (several levels from minimal up to high); **Claude Opus 4.8** has adaptive extended thinking with an *effort* setting (low to max - not a fixed token budget); **Gemini 3 Pro** has deep-think with a `thinking_budget`; and **DeepSeek-R1** reasons in visible `<think>` tags with open weights. On the public intelligence indices Opus 4.8 and GPT-5.5 sit neck-and-neck at the top, so the choice is task-shaped, not raw-IQ-shaped.

**📝 `reasoning_model.py Gemini`** - *API*

In [ ]:
from google.genai import types

# Let the model think natively. Just state the goal - no "think step by step".
cfg = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=4096))  # tokens of internal thought

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents="Three trucks carry 12.5 t, 9.8 t, 14.2 t. A bridge limit is 35 t. "
             "Which two trucks can cross together with the most load?",
    config=cfg)
print(resp.text)
# Output: 12.5 + 14.2 = 26.7 t and 9.8 + 14.2 = 24.0 t and 12.5 + 9.8 = 22.3 t;
# Output: the heaviest valid pair is 12.5 t + 14.2 t = 26.7 t (under 35 t).

- **No "think step by step" in the prompt.** The instruction is just the goal; the model supplies its own reasoning.

- **The reasoning dial** controls how much internal reasoning the model spends - the lever that replaces hand-written chains. On Gemini it is `thinking_budget`; on OpenAI it is `reasoning_effort`; on Anthropic it is adaptive thinking plus an effort setting (the older fixed `budget_tokens` was retired on 4.7/4.8). More effort, more thinking, more latency.

- **The reasoning is internal.** You get a clean answer; the model did the chain itself rather than printing one because you asked.

The canonical 2026 mistake: pasting *"You are a world-class expert. Take a deep breath. Think step by step. Show every step."* in front of a reasoning model. It does not raise accuracy (Gemini 3 Pro's high-thinking config has even measured slightly *below* standard on some tasks) and it inflates latency - reasoning traces already run 4k-16k tokens. The fix is to delete the preamble, state the goal, and set the reasoning budget.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Native reasoning models do the chain internally, so the 2026 skill is reading whether the model in front of you reasons natively - if it does, state the goal and dial the budget instead of hand-writing CoT. And "more thinking" is not free accuracy: an active line of 2026 "overthinking" research shows much of a long trace is filler. **Match reasoning depth to task difficulty.**

---

## 🎯 Concept 4: Tree-of-thought - explore, evaluate, pick

### Tree-of-thought - explore, evaluate, pick

When there is no single right path, generate several and judge them.

> ♟ **Analogy**
>
> **A chess grandmaster.** Before moving, they consider three candidate moves, play each forward mentally, judge the resulting position, then commit. Tree-of-thought generates several approaches, scores them, and picks the winner.
> **Where it breaks down:** exploring branches costs real API calls, and the model is also the judge - if its evaluation is biased, it can confidently pick a worse branch. ToT helps most when approaches are genuinely different and judgeable.

**📝 `tree_of_thought.py Gemini`** - *API*

In [ ]:
def tree_of_thought(problem: str, k: int = 3) -> str:
    """Generate k approaches (diverse), then judge and pick the best (careful)."""
    ideas = ask(f"Problem: {problem}\nPropose {k} DIFFERENT approaches, each reasoned "
                f"to a conclusion. Label them Approach 1..{k}.",
                temperature=0.7)               # high temp -> diverse ideas
    verdict = ask(f"Problem: {problem}\n\nApproaches:\n{ideas}\n\n"
                  "Evaluate each for correctness and feasibility, then pick the best. "
                  "End with 'Best: <number>' and a one-line reason.",
                  temperature=0.1)             # low temp -> careful judgment
    return verdict

plan = "A startup has a small budget and 3 months to ship an MVP. Strategy?"
print(tree_of_thought(plan))
# Output: Approach 1 (build from scratch)... Approach 2 (no-code)...
# Output: Approach 3 (fork open-source)... Best: 3 - fastest to a testable MVP.

- **Call 1 generates breadth at `temperature=0.7`** - high temperature deliberately produces *different* approaches rather than three rewordings of one idea.

- **Call 2 judges at `temperature=0.1`** - low temperature for careful, stable evaluation. Generating wants diversity; judging wants consistency, so the temperatures differ on purpose.

- **The "Best: <number>" marker** lets you parse the winning branch programmatically, the same parse-a-marker pattern from Steps 1-2.

- Two calls, not one: ToT is more expensive than CoT, which is why you reserve it for open-ended problems with several valid paths.

#### 💡 What just happened

⚡ What just happened?Tree-of-thought separates *generating* options (broad, hot) from *judging* them (careful, cold). For open-ended planning and design - where there is no single correct chain - exploring and evaluating several branches beats committing to the first idea. **Use it when the problem has many valid approaches, not one.**

---

## 🎯 Concept 5: ReAct and reflexion - reason with tools, and self-correct

### ReAct and reflexion - reason with tools, and self-correct

Two loops: think-act-observe with tools, and generate-critique-improve.

> 🕵️ **Analogy**
>
> **A detective.** They do not just sit and deduce - they think ("who was here?"), act (check the cameras), observe (three people), think again ("B is suspicious"), act (interview B), and close the case. **ReAct = reasoning + acting** in a loop until confident.
> **Where it breaks down:** a loop with no step limit or no tool-error handling hangs or spirals - the model can keep "thinking" forever or call a tool that fails. Production ReAct needs a max-steps guard and robust parsing.

**📝 `react_loop.py Gemini`** - *API*

In [ ]:
def calc(expr: str) -> str:
    try: return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception: return "calc error"

def react(question: str, max_steps: int = 4) -> str:
    """Think -> Act(calc) -> Observe, looping until 'Final:' or max_steps."""
    history = f"""Answer using the tool calc(expr). Each turn output either
'Action: calc(<expr>)' or 'Final: <answer>'.
Question: {question}"""
    for _ in range(max_steps):                       # max_steps guard = no infinite loop
        out = ask(history, temperature=0.1)
        if "Final:" in out:
            return out.split("Final:")[-1].strip()
        act = re.search(r"calc\((.+?)\)", out)
        obs = calc(act.group(1)) if act else "no tool call"
        history += f"\n{out}\nObservation: {obs}"      # feed result back in
    return "stopped: max steps"

print(react("What is 50000 plus 18% GST on it?"))
# Output: 59000
# (react() returns the text after 'Final:'; internally it ran
#  calc(50000*0.18)->9000.0, then calc(50000+9000)->59000)

- **The model emits an Action or a Final.** Each turn it either calls `calc(...)` or declares it is done - a tiny protocol the prompt defines.

- **The tool runs in your code, not the model.** `calc()` does the exact arithmetic the model is unreliable at, and its result is fed back as an "Observation".

- **Security caveat (this is demo-only):** `eval()` on model-produced text is a code-execution risk, and `{"__builtins__":{}}` does *not* make it safe. In production, never `eval()` model output - use a restricted arithmetic parser (an `ast`-based or math-only evaluator).

- **`max_steps` is the safety rail.** Without it a confused loop runs forever; this is the single most important production guard for ReAct.

- This minimal loop is the seed of agents - Module 11 replaces `calc` with search, databases, and APIs, and adds memory and planning.

**Reflexion** is the other loop: instead of acting with tools, the model critiques its own output and rewrites - generate, critique, improve, repeat until good enough (Shinn et al. 2023). It is ReAct's introspective sibling: great for writing and code, where the model can judge its own work. Both share the same shape - a loop that feeds a result back in - which is why they live in one step.

#### 💡 What just happened

⚡ What just happened?Two loops, one idea: feed a result back into the next turn. **ReAct** loops think-act-observe with external tools (the agent foundation); **reflexion** loops generate-critique-improve with the model judging itself. Both need a stop condition - a max-steps guard or a target score - or they never terminate. **This is where reasoning becomes agentic (Module 11).**

---

## 🎯 Concept 6: Choose and verify - framework, cost, and chain-of-verification

### Choose and verify - framework, cost, and chain-of-verification

Pick the cheapest technique that works, then check the answer before you trust it.

**📝 `chain_of_verification.py`** - *Production*

In [ ]:
# Chain-of-Verification (CoVe): draft -> plan checks -> verify independently -> revise
draft = ask("List 5 key sections of the Indian IT Act, 2000, with section numbers.")

checks = ask(f"Write 5 yes/no fact-check questions about the claims here:\n{draft}")

# Verify WITHOUT showing the draft - so the model cannot just agree with itself
verified = ask(f"Answer each question independently and factually:\n{checks}")

final = ask(f"Original answer:\n{draft}\n\nVerified facts:\n{verified}\n\n"
            "Rewrite the answer, dropping or correcting any claim the facts do not support.")
print(final)
# Output: a revised list with the unverifiable / wrong section numbers removed

- **Step 1 drafts** - the normal (possibly hallucinated) answer.

- **Step 2 plans the checks** - turns the draft's claims into discrete fact-check questions.

- **Step 3 verifies independently** - and crucially does *not* see the draft, so it answers from scratch instead of rubber-stamping the original. This independence is the whole mechanism (it cut hallucinated entities from 2.95 to 0.68 in the paper). The limit: it is the *same* model checking itself, so it cannot supply facts it does not reliably know - for statutes, medical, or financial numbers you still need an external source of truth (RAG or a database, Module 8).

- **Step 4 revises** - reconciles draft against verified facts and drops unsupported claims.

Cost is 3-4 calls per answer, so reserve CoVe for high-stakes factual output - legal citations, statute numbers, medical or financial figures - where a confident wrong answer is expensive.

> 💡 **Info**
>
> The decision framework: start cheap, escalate
> - **Single fact or number?** A standard call, or CoT if it is multi-step. Cheapest.
> - **High-stakes discrete answer?** Self-consistency (sample and vote).
> - **Many valid approaches?** Tree-of-thought (explore and judge).
> - **Needs live data or exact computation?** ReAct (think-act-observe).
> - **Quality-sensitive writing or code?** Reflexion (critique and improve).
> - **Genuinely hard reasoning?** A native reasoning model with a tuned budget.
> - **Factual and risky?** Add chain-of-verification on top.

#### 💡 What just happened

⚡ What just happened?The meta-skill is matching technique to task and spending the least that works: try a standard model, add CoT, escalate to multi-call patterns or a reasoning model only when needed, and verify anything factual before you trust it. **Reasoning is a dial, not a default - and verification is the seatbelt.**

## Putting it together: the reasoning decision map

| Technique | Best for | Calls | Relative cost |
|---|---|---|---|
| Chain-of-thought | Multi-step math and logic, single answer | 1 | Lowest |
| Self-consistency | High-stakes discrete answers | N | N x CoT |
| Reasoning model | Genuinely hard reasoning | 1 | High (long traces) |
| Tree-of-thought | Open-ended, many valid paths | 2-4 | Medium |
| ReAct | Needs tools or live data | 3-6 | Medium-high |
| Reflexion | Quality-sensitive writing or code | 3-9 | Highest |
| Chain-of-verification | Factual, risky output | +3-4 | Add-on |

### 🧮 The whole lesson on one screen

**Make thinking visible** (CoT); **vote when it matters** (self-consistency); **let reasoning models reason** and do not hand-write their chains; **explore when there is no single path** (tree-of-thought); **act and self-correct** for agentic and quality tasks (ReAct, reflexion); and **verify before you trust** (chain-of-verification). Above all: reasoning is a dial - start with the cheapest technique that works and escalate only when it fails.

Forward hooks you just planted: context engineering comes next in Lesson 5.4; we'll parse the answer tag natively in Lesson 5.5 (structured output); prompt optimization is in Lesson 5.6; ReAct and reflexion grow up in Module 11, where we'll build full agents on top of Module 10 tool use; and we'll come back to reasoning cost in Module 13 and to evaluating reasoning quality in Module 14.

- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning* (2022) - [arxiv.org/abs/2201.11903](https://arxiv.org/abs/2201.11903); Kojima et al., *Zero-Shot Reasoners* (2022) - [arxiv.org/abs/2205.11916](https://arxiv.org/abs/2205.11916)

- Wang et al., *Self-Consistency Improves Chain of Thought* (2023) - [arxiv.org/abs/2203.11171](https://arxiv.org/abs/2203.11171)

- Yao et al., *Tree of Thoughts* (2023) - [arxiv.org/abs/2305.10601](https://arxiv.org/abs/2305.10601); Yao et al., *ReAct* (2022) - [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)

- Shinn et al., *Reflexion* (2023) - [arxiv.org/abs/2303.11366](https://arxiv.org/abs/2303.11366)

- Dhuliawala et al., *Chain-of-Verification Reduces Hallucination* (Meta, 2023) - [arxiv.org/abs/2309.11495](https://arxiv.org/abs/2309.11495)

- Snell et al., *Scaling LLM Test-Time Compute* (2024) - [arxiv.org/abs/2408.03314](https://arxiv.org/abs/2408.03314); DeepSeek-AI, *DeepSeek-R1* (2025) - [arxiv.org/abs/2501.12948](https://arxiv.org/abs/2501.12948)

- Reasoning docs: OpenAI [reasoning_effort](https://platform.openai.com/docs/guides/reasoning), Anthropic [extended thinking](https://platform.claude.com/docs/en/build-with-claude/extended-thinking), Google [thinking](https://ai.google.dev/gemini-api/docs/thinking)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 patterns
> - **Chain-of-thought** - make the steps visible; structured CoT separates reasoning from the answer.
> - **Self-consistency** - sample N chains at temperature > 0 and take the majority vote, for discrete answers.
> - **Reasoning models** - they think natively; state the goal and dial the budget, do not hand-write CoT.
> - **Tree-of-thought** - generate diverse approaches (hot), judge them (cold), pick the best.
> - **ReAct and reflexion** - loops that feed a result back: act-with-tools, and critique-and-improve. Bound them.
> - **Choose and verify** - match technique to task, start cheap and escalate, and verify factual output (CoVe).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Advanced Reasoning- CoT, Self-Consistency, and Knowing When to Stop**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.3.html` - regenerate this notebook whenever the source HTML is updated.*
